# Section C — Analytical Queries
All 5 queries with SQL, execution time, and result sets.

In [1]:
import sys, time
from pathlib import Path
import pandas as pd
import psycopg2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, str(Path('..').resolve()))
DSN = 'host=localhost dbname=ecommerce user=postgres password=admin port=5432'

def run_query(conn, sql, label=''):
    t0 = time.time()
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    elapsed = round(time.time() - t0, 3)
    df = pd.DataFrame(rows, columns=cols)
    print(f'{label}  ->  {elapsed}s  ({len(df)} rows)')
    return df, elapsed

conn = psycopg2.connect(DSN)

## Q1 — Funnel Analysis
For each product category: event counts at view / cart / purchase stages + conversion rates.

In [2]:
Q1 = """
SELECT
    dp.category_code,
    COUNT(CASE WHEN fe.event_type = 'view'     THEN 1 END) AS views,
    COUNT(CASE WHEN fe.event_type = 'cart'     THEN 1 END) AS carts,
    COUNT(CASE WHEN fe.event_type = 'purchase' THEN 1 END) AS purchases,
    ROUND(
        100.0 * COUNT(CASE WHEN fe.event_type = 'cart' THEN 1 END)
              / NULLIF(COUNT(CASE WHEN fe.event_type = 'view' THEN 1 END), 0), 2
    ) AS view_to_cart_pct,
    ROUND(
        100.0 * COUNT(CASE WHEN fe.event_type = 'purchase' THEN 1 END)
              / NULLIF(COUNT(CASE WHEN fe.event_type = 'cart' THEN 1 END), 0), 2
    ) AS cart_to_purchase_pct
FROM fact_events fe
JOIN dim_products dp ON fe.product_id = dp.product_id
WHERE dp.category_code IS NOT NULL
GROUP BY dp.category_code
ORDER BY views DESC
LIMIT 10
"""
q1_df, q1_time = run_query(conn, Q1, 'Q1 Funnel')
q1_df

Q1 Funnel  ->  3.442s  (10 rows)


,category_code,views,carts,purchases,view_to_cart_pct,cart_to_purchase_pct
0,electronics.smartphone,3464814,156948,104840,4.53,66.80
1,electronics.clocks,431970,5805,5810,1.34,100.09
2,computers.notebook,386638,4285,4899,1.11,114.33
3,electronics.audio.headphone,334434,15161,9491,4.53,62.60
4,electronics.video.tv,332659,8564,6525,2.57,76.19
5,appliances.kitchen.refrigerators,274756,1649,3139,0.60,190.36
6,appliances.kitchen.washer,268682,3766,4824,1.40,128.09
7,appliances.environment.vacuum,254728,3513,3737,1.38,106.38
8,apparel.shoes,199011,0,870,0.00,None
9,auto.accessories.player,176076,451,1708,0.26,378.71


## Q2 — Session Aggregation
Per session: total events, distinct products, total cart value, purchase flag.

In [3]:
Q2 = """
SELECT user_session, user_id, COUNT(*) AS total_events,
       COUNT(DISTINCT product_id) AS distinct_products,
       ROUND(SUM(CASE WHEN event_type = 'cart' THEN price ELSE 0 END)::numeric, 2) AS total_cart_value,
       MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)::boolean AS purchased
FROM (SELECT * FROM fact_events ORDER BY user_session LIMIT 200000) sub
GROUP BY user_session, user_id
ORDER BY total_events DESC
LIMIT 10
"""
q2_df, q2_time = run_query(conn, Q2, 'Q2 Sessions')
q2_df

Q2 Sessions  ->  4.378s  (10 rows)


,user_session,user_id,total_events,distinct_products,total_cart_value,purchased
0,010fe720-ec78-4311-9562-dd680bfbc37e,526012229,252,55,0.00,False
1,01e3e4af-1c4b-4ad2-a026-e96305563722,546309572,210,91,0.00,False
2,0132dea5-8cac-4142-9f3d-0d7d2a1dfdd4,539941524,202,49,154.38,True
3,0281b173-80df-434f-a953-c06121624280,519433003,177,96,0.00,False
4,03175d5f-ea3e-4dd2-9e7b-e255222a8b94,513900355,133,64,5869.24,True
5,028f2848-78c5-42ac-95f3-ddacdd5e4607,531513623,129,48,0.00,False
6,00fcaeba-e78d-4abf-b7fb-0d57a6644755,523781231,128,80,0.00,False
7,02bdd270-f4b8-4ef9-a5a5-d9bff394a379,543346576,116,55,0.00,True
8,0071ad9d-6aa6-4ab6-b50c-ef65dd1c1144,546270188,105,41,0.00,False
9,0184d4d0-30f1-403c-afa3-2a08b54bb7bb,556250487,104,44,0.00,True


## Q3 — Top 10 Brands by Revenue per Month

In [4]:
Q3 = """
WITH ranked AS (
    SELECT
        dp.brand,
        fe.source_month,
        ROUND(SUM(fe.price)::numeric, 2) AS total_revenue,
        RANK() OVER (PARTITION BY fe.source_month ORDER BY SUM(fe.price) DESC) AS rnk
    FROM fact_events fe
    JOIN dim_products dp ON fe.product_id = dp.product_id
    WHERE fe.event_type = 'purchase' AND dp.brand IS NOT NULL
    GROUP BY dp.brand, fe.source_month
)
SELECT brand, source_month, total_revenue, rnk
FROM ranked
WHERE rnk <= 10
ORDER BY source_month, rnk
"""
q3_df, q3_time = run_query(conn, Q3, 'Q3 Top Brands')
q3_df

Q3 Top Brands  ->  1.316s  (20 rows)


,brand,source_month,total_revenue,rnk
0,apple,2019-10,24967113.79,1
1,samsung,2019-10,10541912.75,2
2,xiaomi,2019-10,2000501.03,3
3,huawei,2019-10,1194021.02,4
4,acer,2019-10,815328.44,5
5,lg,2019-10,780938.99,6
6,lucente,2019-10,675345.37,7
7,sony,2019-10,537785.92,8
8,oppo,2019-10,488830.13,9
9,lenovo,2019-10,483721.95,10


## Q4 — Users Who Purchased in Oct but Not in Nov

In [5]:
Q4 = """
SELECT user_id FROM fact_events
WHERE event_type = 'purchase' AND source_month = '2019-10'
EXCEPT
SELECT user_id FROM fact_events WHERE source_month = '2019-11'
ORDER BY user_id LIMIT 10
"""
q4_df, q4_time = run_query(conn, Q4, 'Q4 Oct-only buyers')
q4_df

Q4 Oct-only buyers  ->  2.193s  (10 rows)


,user_id
0,264649825
1,340041246
2,420935067
3,433754231
4,435648894
5,437371552
6,438992161
7,439111345
8,440471930
9,440756116


## Q5 — Hourly Distribution of Purchase Events

In [6]:
Q5 = """
SELECT
    EXTRACT(HOUR FROM event_time)::int AS hour_of_day,
    COUNT(*) AS purchase_count
FROM fact_events
WHERE event_type = 'purchase'
GROUP BY hour_of_day
ORDER BY purchase_count DESC
"""
q5_df, q5_time = run_query(conn, Q5, 'Q5 Hourly')
q5_df.head(10)

Q5 Hourly  ->  2.473s  (24 rows)


,hour_of_day,purchase_count
0,9,18130
1,8,17615
2,7,17457
3,6,17097
4,10,17070
5,5,15700
6,11,15110
7,4,13220
8,12,12554
9,13,11672


## Query Execution Time Summary

In [7]:
timing_df = pd.DataFrame([
    {'query': 'Q1 Funnel',     'time_s': q1_time},
    {'query': 'Q2 Sessions',   'time_s': q2_time},
    {'query': 'Q3 Top Brands', 'time_s': q3_time},
    {'query': 'Q4 Oct-only',   'time_s': q4_time},
    {'query': 'Q5 Hourly',     'time_s': q5_time},
])
print(timing_df.to_string(index=False))

        query  time_s
    Q1 Funnel   3.442
  Q2 Sessions   4.378
Q3 Top Brands   1.316
  Q4 Oct-only   2.193
    Q5 Hourly   2.473


## D1 — Visualisation: Conversion Funnel (Top 10 Categories)

In [8]:
top10 = q1_df.head(10).copy()
top10['category_short'] = top10['category_code'].str.split('.').str[-1]

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(top10))
w = 0.25
ax.bar([i - w for i in x], top10['views'],     w, label='View',     color='#3498db')
ax.bar([i       for i in x], top10['carts'],   w, label='Cart',     color='#f39c12')
ax.bar([i + w for i in x], top10['purchases'], w, label='Purchase', color='#27ae60')
ax.set_xticks(list(x))
ax.set_xticklabels(top10['category_short'], rotation=30, ha='right')
ax.set_ylabel('Event Count')
ax.set_title('Conversion Funnel — Top 10 Categories')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/funnel_chart.png', dpi=150)
plt.show()

C:\Users\Yash Sharma\AppData\Local\Temp\ipykernel_19984\737069442.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
conn.close()